In [3]:
import os

In [5]:
%pwd

'f:\\UOM\\FYP\\MLFlow_project\\research'

In [6]:
os.chdir('../')

In [7]:
%pwd

'f:\\UOM\\FYP\\MLFlow_project'

In [8]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [2]:
from box import Box, ConfigBox
print(Box, ConfigBox)



<class 'box.box.Box'> <class 'box.config_box.ConfigBox'>


In [10]:
from pathlib import Path
import sys, os

# Ensure we're using the project root and add the absolute `src` path to sys.path
project_root = Path.cwd()
src_path = str(project_root / "src")
print("CWD:", project_root)
print("Adding to sys.path:", src_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Quick verification
import pkgutil
print("Mlflow_project present in src?", any(p.name == 'Mlflow_project' for p in pkgutil.iter_modules([src_path])))

# Now import
from Mlflow_project.constants import *
from Mlflow_project.utils.common import read_yaml, create_directories
print("Imported Mlflow_project successfully.")

CWD: f:\UOM\FYP\MLFlow_project
Adding to sys.path: f:\UOM\FYP\MLFlow_project\src
Mlflow_project present in src? True
Imported Mlflow_project successfully.


In [ ]:
from pathlib import Path
import yaml

# Ensure params.yaml exists and is non-empty (creates sensible defaults if not)
params_path = Path("params.yaml")
if not params_path.exists() or params_path.stat().st_size == 0:
    default_params = {
        "model": {
            "seed": 42,
            "n_estimators": 100
        },
        "data": {
            "test_size": 0.2
        }
    }
    with params_path.open("w") as f:
        yaml.dump(default_params, f)
    print(f"Created default {params_path}")
else:
    print(f"{params_path} already exists and is non-empty")

In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config
      

In [12]:
import os
import zipfile
import gdown
from Mlflow_project import logger
from Mlflow_project.utils.common import get_size

In [13]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
     
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("resources/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix+file_id,zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [20]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-12-23 11:50:31,913: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-12-23 11:50:31,989: INFO: common: yaml file: params.yaml loaded successfully]
[2025-12-23 11:50:31,996: INFO: common: created directory at: resources]
[2025-12-23 11:50:31,998: INFO: common: created directory at: resources/data_ingestion]
[2025-12-23 11:50:32,001: INFO: 3167781939: Downloading data from https://drive.google.com/file/d/1-ocwrtBy9B8fZOQnCPIN50M7Oh3Rmr_v/view?usp=sharing into file resources/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1-ocwrtBy9B8fZOQnCPIN50M7Oh3Rmr_v
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1-ocwrtBy9B8fZOQnCPIN50M7Oh3Rmr_v&confirm=t&uuid=cca2b304-74f0-4500-aa02-e645f9c29215
To: f:\UOM\FYP\MLFlow_project\resources\data_ingestion\data.zip
100%|██████████| 156M/156M [01:31<00:00, 1.71MB/s] 

[2025-12-23 11:52:08,069: INFO: 3167781939: Downloaded data from https://drive.google.com/file/d/1-ocwrtBy9B8fZOQnCPIN50M7Oh3Rmr_v/view?usp=sharing into file resources/data_ingestion/data.zip]
